# Goal 52: H100 2B Production Readiness

This notebook builds a local code tokenizer and packed shards, then gates 304M, 1B, and 2B H100 admission probes. It never promotes a larger run unless the prior staged probe passes. Checkpoints stay outside Git; the final bundle contains lightweight JSON evidence only.

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys, torch

REPO_URL = 'https://github.com/NikanBHMNJ/MoP.git'
REPO = Path('/content/MoP')
if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO)], check=True)
os.chdir(REPO)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[dev,gpu,hf]'], check=True)
subprocess.run(['mopforge', 'doctor'], check=True)
subprocess.run(['mopforge', 'runtime', 'detect'], check=True)

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError('Goal 52 requires a CUDA H100 runtime.')
props = torch.cuda.get_device_properties(0)
total_gb = props.total_memory / (1024 ** 3)
if 'H100' not in props.name.upper():
    raise RuntimeError(f'Expected H100, detected {props.name}.')
if 75 <= total_gb <= 86:
    memory_tier = 80
elif 86 < total_gb <= 100:
    memory_tier = 94
else:
    raise RuntimeError(f'Unsupported H100 memory tier: {total_gb:.2f} GiB.')
CONFIG_300M = Path('configs/jobs/goal52_300m_dense_h100_probe.json')
CONFIG_1B = Path('configs/jobs/goal52_1b_dense_h100_probe.json')
CONFIG_2B = Path(f'configs/jobs/goal52_2b_dense_h100_{memory_tier}gb_probe.json')
REPORT_DIR = Path('reports/goal52_h100_2b_readiness')
REPORT_DIR.mkdir(parents=True, exist_ok=True)
print({'gpu': props.name, 'memory_gib': round(total_gb, 2), 'tier': memory_tier})

## Data
Set `CORPUS_PATH` to a trusted UTF-8 text or JSONL code corpus. JSONL records must use the `text` field. Dataset licensing, provenance, deduplication, and contamination review remain the operator's responsibility.

In [ ]:
CORPUS_PATH = Path('/content/code_corpus.jsonl')
MAX_RECORDS = None
if not CORPUS_PATH.is_file():
    raise FileNotFoundError(f'Upload or mount the code corpus at {CORPUS_PATH}.')
TOKENIZER_DIR = Path('data/goal52_tokenizer')
TOKEN_DIR = Path('data/goal52_code_tokens')
TOKENIZER_SPEC = TOKENIZER_DIR / 'tokenizer_spec.json'
TOKEN_MANIFEST = TOKEN_DIR / 'manifest.json'
if not TOKENIZER_SPEC.exists():
    command = ['mopforge', 'tokenizer', 'train-bpe', str(CORPUS_PATH), '--output-dir', str(TOKENIZER_DIR), '--vocab-size', '32768', '--min-frequency', '2', '--text-field', 'text']
    if MAX_RECORDS is not None: command += ['--max-records', str(MAX_RECORDS)]
    subprocess.run(command, check=True)
if not TOKEN_MANIFEST.exists():
    command = ['mopforge', 'gpu', 'pack-corpus', str(CORPUS_PATH), '--tokenizer-spec', str(TOKENIZER_SPEC), '--output-dir', str(TOKEN_DIR), '--sequence-length', '1024', '--tokens-per-shard', '10000000', '--eval-fraction', '0.01', '--split-seed', '42', '--text-field', 'text']
    if MAX_RECORDS is not None: command += ['--max-records', str(MAX_RECORDS)]
    subprocess.run(command, check=True)
manifest = json.loads(TOKEN_MANIFEST.read_text())
print({'tokenizer_spec': str(TOKENIZER_SPEC), 'manifest': str(TOKEN_MANIFEST), 'splits': manifest.get('splits')})

In [ ]:
for config in (CONFIG_300M, CONFIG_1B, CONFIG_2B):
    subprocess.run(['mopforge', 'gpu', 'validate', str(config)], check=True)
    subprocess.run(['mopforge', 'gpu', 'estimate', str(config)], check=True)

In [ ]:
RUN_300M = True
REPORT_300M = REPORT_DIR / '300m_probe.json'
if RUN_300M:
    subprocess.run(['mopforge', 'gpu', 'probe', str(CONFIG_300M), '--optimizer-updates', '20', '--output', str(REPORT_300M)], check=False)
if not REPORT_300M.exists():
    raise RuntimeError('The 300M probe report is required before promotion.')
gate_300m = json.loads(REPORT_300M.read_text()).get('acceptance', {})
if not gate_300m.get('passed'):
    raise RuntimeError(f'300M admission failed: {gate_300m}')
print('300M gate passed')

In [ ]:
RUN_1B = True
REPORT_1B = REPORT_DIR / '1b_probe.json'
if RUN_1B:
    subprocess.run(['mopforge', 'gpu', 'probe', str(CONFIG_1B), '--optimizer-updates', '20', '--output', str(REPORT_1B)], check=False)
if not REPORT_1B.exists():
    raise RuntimeError('The 1B probe report is required before 2B promotion.')
gate_1b = json.loads(REPORT_1B.read_text()).get('acceptance', {})
if not gate_1b.get('passed'):
    raise RuntimeError(f'1B admission failed: {gate_1b}')
print('1B gate passed')

In [ ]:
RUN_2B = True
REPORT_2B = REPORT_DIR / '2b_probe.json'
if RUN_2B:
    subprocess.run(['mopforge', 'gpu', 'probe', str(CONFIG_2B), '--optimizer-updates', '20', '--output', str(REPORT_2B)], check=False)
if not REPORT_2B.exists():
    raise RuntimeError('The 2B probe did not produce a report.')
gate_2b = json.loads(REPORT_2B.read_text()).get('acceptance', {})
if not gate_2b.get('passed'):
    raise RuntimeError(f'2B admission failed: {gate_2b}')
print('2B admission gate passed')

## Multi-H100 Pilot
The Colab notebook does not launch the eight-GPU pilot. Run the printed command on an eight-H100 node only after the 2B admission report passes.

In [ ]:
PILOT_CONFIG = Path('configs/jobs/goal52_2b_dense_8xh100_fsdp_pilot.json')
subprocess.run(['mopforge', 'gpu', 'launch-torchrun', str(PILOT_CONFIG), '--dry-run'], check=True)

In [ ]:
summary = {
    'format': 'mopforge_goal52_h100_readiness_v1',
    'gpu': props.name,
    'memory_gib': round(total_gb, 4),
    'memory_tier': memory_tier,
    'token_manifest': str(TOKEN_MANIFEST),
    'tokenizer_spec': str(TOKENIZER_SPEC),
    'gates': {'300m': gate_300m, '1b': gate_1b, '2b': gate_2b},
}
(REPORT_DIR / 'readiness_summary.json').write_text(json.dumps(summary, indent=2, sort_keys=True))
archive = shutil.make_archive('/content/goal52_h100_reports', 'zip', REPORT_DIR)
print({'report_archive': archive, 'contains_weights': False})
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    pass